In [1]:
import os
import sys

# add the source directory to system path, so that relative imports work
# this fix is only for Jupyter Notebooks
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
from common.constants import ALTAIRA_MU as MU, ALTAIRA_AU as AU, YEAR
from common.solar_sail import calc_opt_un
from common.constants import solar_C, solar_r0, solar_A, sc_mass

In [3]:
import math
import numpy as np

from scipy.integrate import solve_ivp

In [4]:
r0x = -200*AU
r0y = -0.2*AU
r0z = 0

r0 = np.array([r0x, r0y, r0z])
v0 = np.array([50.0, 0, 0])

print(r0,v0)

[-2.99195741e+10 -2.99195741e+07  0.00000000e+00] [50.  0.  0.]


In [5]:
def dydt(t,y):
    r = y[0:3]; v = y[3:6]
    dr = v; dv = -MU/np.linalg.norm(r)**3 * r
    return np.concatenate([dr, dv])

In [6]:
def event(t,y):
    # periapsis
    r = y[0:3]; v = y[3:6]
    # return np.dot(r,v)
    return np.linalg.norm(r) - 5*AU

event.terminal = True

In [7]:
res1 = solve_ivp(dydt, [0, 20*YEAR], np.concatenate([r0, v0]), events=event, method='DOP853', rtol=1e-16, atol=1e-16)

print(res1['t'][-1] / YEAR)
print(res1['y'][0:3,-1] / AU, res1['y'][3:6,-1])
print(np.linalg.norm(res1['y'][0:3,-1] / AU))
print(event(res1['t'][-1], res1['y'][:,-1]))

18.395331592535342
[-4.99625879 -0.19338586  0.        ] [53.50956938  0.06965097  0.        ]
4.999999999999981
-2.7418136596679688e-06


/home/aryad/miniforge3/envs/gtoc13/lib/python3.13/site-packages/scipy/integrate/_ivp/rk.py:505: UserWarning: At least one element of `rtol` is too small. Setting `rtol = np.maximum(rtol, 2.220446049250313e-14)`.
  super().__init__(fun, t0, y0, t_bound, max_step, rtol, atol,


In [8]:
print(res1['t'][-1])
print(res1['y'][0:3,-1], res1['y'][3:6,-1])
print(np.linalg.norm(res1['y'][0:3,-1]))
print(event(res1['t'][-1], res1['y'][:,-1]))

580512516.2645934
[-7.47429677e+08 -2.89301134e+07  0.00000000e+00] [53.50956938  0.06965097  0.        ]
747989353.4549973
-2.7418136596679688e-06


In [9]:
def dydt2(t,y):
    r = y[0:3]; v = y[3:6]

    # calculate control vector
    ur = -r; ud = -v
    ur = ur/np.linalg.norm(ur);
    ud = ud/np.linalg.norm(ud)
    un = calc_opt_un(ur,ud)

    # calculate the force of the solar sail
    r_mag = np.linalg.norm(r)
    s_mag = 2*(solar_C*solar_A/sc_mass) * (solar_r0/r_mag)**2 * np.dot(ur,un)**2
    s_mag = s_mag / 1000    # convert from m/s^2 to km/s^2
    s_acc = -s_mag * un   # the acceleration acts opposite to un

    dy = np.empty((6,), dtype=np.float64)
    dr = v; dv = (-MU/r_mag**3 * r) + s_acc   # gravity and solar acceleration
    dy[0:3] = dr; dy[3:6] = dv

    return dy

In [10]:
# r02 = np.array([0.0295637,  -0.03189925,  0.])*AU
# v02 = np.array([168.93454214, 156.56573665,   0.])
r02 = res1['y'][0:3,-1]
v02 = res1['y'][3:6,-1]

In [11]:
def event2(t,y):
    r = y[0:3]; v = y[3:6]
    return np.linalg.norm(r) - 7*AU

event2.terminal = True

In [12]:
def event3(t,y):
    r = y[0:3]; v = y[3:6]
    return np.dot(r,v)
# event3.terminal = True

In [13]:
def energy(r,v,mu):
    r = np.linalg.norm(r)
    v = np.linalg.norm(v)
    return v**2/2 - MU/r

In [14]:
res2 = solve_ivp(dydt2, [res1['t'][-1], res1['t'][-1] + 1*YEAR], np.concatenate([r02, v02]),
                 events=[event2,event3], method='DOP853', rtol=1e-16, atol=1e-16)

print(res2['t'][-1] / YEAR)
print(res2['y'][0:3,-1] / AU, res2['y'][3:6,-1])
print(np.linalg.norm(res2['y'][0:3,-1] / AU))
print(energy(res2['y'][0:3,-1], res2['y'][3:6,-1], MU))

19.395331592535342
[-2.31680709  1.38927348  0.        ] [-6.94813807  0.16637711  0.        ]
2.7014210908012997
-320.66053352339867


In [15]:
print(res2['t'][-1])
print(res2['y'][0:3,-1], res2['y'][3:6,-1])
print(np.linalg.norm(res2['y'][0:3,-1]))
print(energy(res2['y'][0:3,-1], res2['y'][3:6,-1], MU))

612070116.2645934
[-3.46589408e+08  2.07832355e+08  0.00000000e+00] [-6.94813807  0.16637711  0.        ]
404126843.0236331
-320.66053352339867


In [16]:
np.linalg.norm(np.array(res2['y_events'][1])[0:3] / AU)

np.float64(0.048929188308089525)

In [17]:
res2['y_events']

[array([], dtype=float64),
 array([[ 6.63771347e+06, -3.08525572e+06,  0.00000000e+00,
          8.21574320e+01,  1.76756010e+02,  0.00000000e+00]])]

In [18]:
event3(res2['t'][-1], res2['y'][:,-1])

np.float64(2442729607.9566183)

In [19]:
print(res2['t'][-1] + 580512516.2646053)

1192582632.5291986


In [20]:
calc_opt_un(-r02,-v02)

array([0.99980012, 0.01999291, 0.        ])

In [21]:
calc_opt_un(-np.array([-2.94967297e+08,  1.55202225e+08,  0.00000000e+00]), -np.array([-1.54683089, -3.79199229,  0.        ]))

array([ 0.34248393, -0.93952369,  0.        ])

In [22]:
# remove rows with time steps < 60s
mask = np.diff(res2['t']) > 60
mask = np.concatenate([[True], mask])
mask[0] = True; mask[-1] = True

res2t = res2['t'][mask]
res2y = res2['y'][:,mask]

In [23]:
# first sol:
sol_line_1_raw = [
    '0', '0', '0.0', *r0, *v0, [0.0, 0.0, 0.0]
]

sol_line_2_raw = [
    '0', '0', res1['t'][-1], *res1['y'][:,-1], [0.0, 0.0, 0.0]
]

In [24]:
def flatten_list(l):
    out = []
    for v in l:
        if isinstance(v, (int, float, str)):
            out.append(float(v))
        elif isinstance(v, (list, np.ndarray)):
            for y in v:
                out.append(float(y))
    return out

In [25]:
sol_lines = list()
sol_lines.append(flatten_list(sol_line_1_raw))
sol_lines.append(flatten_list(sol_line_2_raw))

# propagated arc
for i in range(len(res2t)):
    ti = res2t[i]; yi = res2y[:,i]
    sol_line_raw = ['0', '1', ti, *yi, *calc_opt_un(-yi[0:3], -yi[3:6])]
    sol_lines.append(flatten_list(sol_line_raw))

In [26]:
import csv

with open('capture_test.csv', 'w', newline='') as f:
    header = ['#body_id','flag','epoch','rx','ry','rz','vx','vy','vz','ux','uy','uz']
    writer = csv.writer(f)
    writer.writerow(header)
    for l in sol_lines:
        writer.writerow(l)

In [27]:
np.diff(res2['t']).mean()

np.float64(119085.28301886792)